# 01 - Data Exploration

Load synthetic PR data, inspect shapes and dtypes, examine the distribution
of PR outcomes, and identify the top repos by volume.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from oss_pulse.visualize.style import setup_style, PALETTE, save_fig

setup_style()

In [ ]:
# Load processed data produced by `make demo`
DATA_DIR = Path("../data/processed")

pr_df = pd.read_parquet(DATA_DIR / "pr_events_featured.parquet")
repos_df = pd.read_parquet(Path("../data/raw/top_repos.parquet"))
monthly_df = pd.read_parquet(DATA_DIR / "repo_monthly.parquet")
weekly_df = pd.read_parquet(DATA_DIR / "repo_weekly.parquet")

print(f"PR events:     {pr_df.shape}")
print(f"Repos:         {repos_df.shape}")
print(f"Monthly agg:   {monthly_df.shape}")
print(f"Weekly agg:    {weekly_df.shape}")

In [ ]:
# Basic dtype inspection
print("--- PR events dtypes ---")
print(pr_df.dtypes)
print("\n--- Descriptive statistics ---")
pr_df.describe(include="all").T

In [ ]:
# Distribution of PR outcomes
outcome_counts = pr_df["pr_outcome"].value_counts()
print("PR outcome distribution:")
print(outcome_counts)

fig, ax = plt.subplots(figsize=(8, 5))
outcome_counts.plot.bar(ax=ax, color=PALETTE["primary"], edgecolor=PALETTE["bg"])
ax.set_title("PR Outcome Distribution", fontsize=14, fontweight="bold")
ax.set_ylabel("Count")
ax.set_xlabel("Outcome")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Top repos by PR volume
top_repos = (
    pr_df.groupby("repo_name")["pr_number"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)
print("Top 10 repos by unique PR count:")
print(top_repos)

fig, ax = plt.subplots(figsize=(10, 6))
top_repos.sort_values().plot.barh(ax=ax, color=PALETTE["success"], edgecolor=PALETTE["bg"])
ax.set_title("Top 10 Repos by PR Volume", fontsize=14, fontweight="bold")
ax.set_xlabel("Unique PRs")
plt.tight_layout()
plt.show()

In [ ]:
# Author class breakdown and bot ratio
print("Author class distribution:")
print(pr_df["author_class"].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pr_df["author_class"].value_counts().plot.pie(
    ax=axes[0], autopct="%1.1f%%", startangle=140,
    colors=[PALETTE["primary"], PALETTE["success"], PALETTE["warning"], PALETTE["accent"]],
)
axes[0].set_ylabel("")
axes[0].set_title("Author Class", fontsize=12, fontweight="bold")

pr_df["pr_size_bucket"].value_counts().plot.bar(
    ax=axes[1], color=PALETTE["warning"], edgecolor=PALETTE["bg"],
)
axes[1].set_title("PR Size Distribution", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.show()